# Disaggregasi Data Tanaman Pangan Kabupaten Kediri → Kecamatan

Notebook ini menjalankan rencana disaggregasi yang kamu pilih:

**Tanaman Pangan Kabupaten Kediri + Luas Sawah per Kecamatan → Proporsi Luas Sawah → Estimasi Luas Panen & Produksi → Produktivitas → Validasi → Dataset Final.**

Target output:
- 26 kecamatan × 5 komoditas = **130 baris**
- Estimasi luas panen per kecamatan
- Estimasi produksi per kecamatan
- Produktivitas hasil estimasi
- Validasi total terhadap data Kabupaten
- Metadata bahwa data kecamatan adalah **Estimated**
- Export Excel dan CSV

**Catatan penting:** program tidak membuat variasi produktivitas secara acak. Karena luas panen dan produksi sama-sama dibagi menggunakan proporsi luas sawah, produktivitas komoditas yang sama akan sama antar kecamatan. Ini adalah konsekuensi matematis dari metode yang kamu pilih.


In [6]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("✓ Program siap dijalankan.")


✓ Program siap dijalankan.


In [21]:
import os
import pandas as pd

# =====================================================================
# 1. PENGECEKAN LOKASI & DAFTAR FILE
# =====================================================================

# Penentuan path folder data secara fleksibel (baik dijalankan dari root maupun dari subfolder notebooks/)
if os.path.exists("../data"):
    dir = "../data"
elif os.path.exists("data"):
    dir = "data"
else:
    dir = os.path.abspath("data")

print(f"Daftar file yang tersedia di folder '{dir}':")

# Menampilkan daftar dan ukuran file
if os.path.exists(dir):
    available_files = sorted(os.listdir(dir))
    for f in available_files:
        f_path = os.path.join(dir, f)
        if os.path.isfile(f_path):
            size_mb = os.path.getsize(f_path) / (1024 * 1024)
            print(f" - {f:<38} : {size_mb:.2f} MB")
else:
    print("Folder data/raw tidak ditemukan. Pastikan path sudah benar.")

# =====================================================================
# 2. MEMUAT DATASET & RINGKASAN DATA
# =====================================================================

# Menentukan lokasi masing-masing file
file_sawah = os.path.join(dir, "Luas Lahan Per Kecamatan.csv")
file_tanaman = os.path.join(dir, "Tanaman Pangan.xlsx")

# Membaca dataset CSV dan Excel
df_sawah = pd.read_csv(file_sawah)
df_tanaman = pd.read_excel(file_tanaman)

# Menampilkan dimensi data
print(f"\nDimensi Dataset Luas Lahan    : {df_sawah.shape[0]:,} baris dan {df_sawah.shape[1]} kolom.")
print(f"Dimensi Dataset Tanaman Pangan: {df_tanaman.shape[0]:,} baris dan {df_tanaman.shape[1]} kolom.\n")

# Menampilkan tipe data masing-masing
print("Ringkasan Tipe Data Dataset Luas Lahan:")
df_sawah.info()
print("-" * 60)

print("\nRingkasan Tipe Data Dataset Tanaman Pangan:")
df_tanaman.info()
print("-" * 60)

# ============================================================
# 3. TAMPILKAN DATA TANAMAN PANGAN KABUPATEN
# ============================================================

print("\nPreview Dataset Tanaman Pangan:")
display(df_tanaman)

Daftar file yang tersedia di folder '../data':
 - .gitkeep                               : 0.00 MB
 - Luas Lahan Per Kecamatan.csv           : 0.00 MB
 - Tanaman Pangan Per Kecamatan.csv       : 0.01 MB
 - Tanaman Pangan.xlsx                    : 0.01 MB

Dimensi Dataset Luas Lahan    : 29 baris dan 5 kolom.
Dimensi Dataset Tanaman Pangan: 5 baris dan 5 kolom.

Ringkasan Tipe Data Dataset Luas Lahan:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Subdistrict  26 non-null     object
 1   Unnamed: 1   29 non-null     object
 2   Unnamed: 2   28 non-null     object
 3   Unnamed: 3   28 non-null     object
 4   Unnamed: 4   28 non-null     object
dtypes: object(5)
memory usage: 1.3+ KB
------------------------------------------------------------

Ringkasan Tipe Data Dataset Tanaman Pangan:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 

,No,Tanaman,Luas Panen,Produksi,Produktivitas
0,1,Padi,34198.23 Ha,198324.32 Ton,57.992568621242 Kuintal/Ha
1,2,Kedelai,1.9 Ha,2.5030348615385 Ton,13.17386769 Kuintal/Ha
2,3,Kacang Tanah,476.5 Ha,843.69645825108 Ton,17.706116647452 Kuintal/Ha
3,4,Ubi Jalar,538.5 Ha,15825.8317 Ton,293.88731104921 Kuintal/Ha
4,5,Ketela Pohon,652.8 Ha,18597.024533333 Ton,284.88089052287 Kuintal/Ha


In [ ]:
# ============================================================
# 4. CLEANING DATA TANAMAN PANGAN
# ============================================================

def extract_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = re.sub(r"[^0-9.,-]", "", value)

    if "," in value and "." in value:
        value = value.replace(",", "")
    elif "," in value:
        value = value.replace(",", ".")

    try:
        return float(value)
    except ValueError:
        return np.nan


required_crop_columns = {"Tanaman", "Luas Panen", "Produksi", "Produktivitas"}
missing_crop = required_crop_columns - set(df_tanaman.columns)

if missing_crop:
    raise ValueError(f"Kolom tanaman pangan tidak ditemukan: {missing_crop}")

df_tanaman = df_tanaman.copy()

df_tanaman["luas_panen_kab_ha"] = df_tanaman["Luas Panen"].apply(extract_number)
df_tanaman["produksi_kab_ton"] = df_tanaman["Produksi"].apply(extract_number)
df_tanaman["produktivitas_kab_kuintal_ha"] = (
    df_tanaman["Produktivitas"].apply(extract_number)
)

df_tanaman = df_tanaman[
    [
        "Tanaman",
        "luas_panen_kab_ha",
        "produksi_kab_ton",
        "produktivitas_kab_kuintal_ha",
    ]
].copy()

df_tanaman["Tanaman"] = df_tanaman["Tanaman"].astype(str).str.strip()

if df_tanaman[["luas_panen_kab_ha", "produksi_kab_ton"]].isna().any().any():
    raise ValueError("Ada nilai luas panen/produksi yang tidak berhasil dibaca sebagai angka.")

print("Komoditas:", df_tanaman["Tanaman"].tolist())
display(df_tanaman)


Komoditas: ['Padi', 'Kedelai', 'Kacang Tanah', 'Ubi Jalar', 'Ketela Pohon']


,Tanaman,luas_panen_kab_ha,produksi_kab_ton,produktivitas_kab_kuintal_ha
0,Padi,"34,198.2300","198,324.3200",57.9926
1,Kedelai,1.9000,2.5030,13.1739
2,Kacang Tanah,476.5000,843.6965,17.7061
3,Ubi Jalar,538.5000,"15,825.8317",293.8873
4,Ketela Pohon,652.8000,"18,597.0245",284.8809


In [ ]:
# ============================================================
# 5. BACA DATA LUAS SAWAH PER KECAMATAN
# ============================================================

# CSV yang kamu upload memiliki 4 baris pembuka sebelum header data.
df_sawah_raw = pd.read_csv(file_sawah, skiprows=4, header=None)

print("Ukuran dataset:", df_sawah_raw.shape)
display(df_sawah_raw.head())


Ukuran dataset: (26, 5)


,0,1,2,3,4
0,[010] Mojo,926,7371,1866,10163
1,[020] Semen,1391,5579,901,7871
2,[030] Ngadiluwih,1160,984,1814,3958
3,[040] Kras,2191,513,1844,4548
4,[050] Ringinrejo,1590,555,2083,4228


In [ ]:
# ============================================================
# 6. CLEANING DATA LUAS SAWAH
# ============================================================

if len(df_sawah_raw.columns) < 5:
    raise ValueError("Struktur CSV luas sawah tidak sesuai.")

df_sawah = df_sawah_raw.iloc[:, :5].copy()

df_sawah.columns = [
    "kecamatan",
    "luas_sawah_ha",
    "lahan_pertanian_non_sawah_ha",
    "lahan_non_pertanian_ha",
    "total_lahan_ha",
]

def clean_kecamatan(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    return re.sub(r"^\[\d+\]\s*", "", value).strip()

df_sawah["kecamatan"] = df_sawah["kecamatan"].apply(clean_kecamatan)

numeric_cols = [
    "luas_sawah_ha",
    "lahan_pertanian_non_sawah_ha",
    "lahan_non_pertanian_ha",
    "total_lahan_ha",
]

for col in numeric_cols:
    df_sawah[col] = pd.to_numeric(df_sawah[col], errors="coerce")

if df_sawah["kecamatan"].isna().any():
    raise ValueError("Ada nama kecamatan yang kosong.")

if df_sawah["luas_sawah_ha"].isna().any():
    raise ValueError("Ada nilai luas sawah yang tidak valid.")

display(df_sawah)


,kecamatan,luas_sawah_ha,lahan_pertanian_non_sawah_ha,lahan_non_pertanian_ha,total_lahan_ha
0,Mojo,926,7371,1866,10163
1,Semen,1391,5579,901,7871
2,Ngadiluwih,1160,984,1814,3958
3,Kras,2191,513,1844,4548
4,Ringinrejo,1590,555,2083,4228
5,Kandat,1984,1048,1841,4873
6,Wates,3370,241,2902,6513
7,Ngancar,1508,4847,1726,8081
8,Plosoklaten,2153,2329,2276,6758
9,Gurah,2272,382,2101,4755


In [ ]:
# ============================================================
# 7. VALIDASI KECAMATAN DAN TOTAL LUAS SAWAH
# ============================================================

jumlah_kecamatan = df_sawah["kecamatan"].nunique()

if jumlah_kecamatan != 26:
    raise ValueError(
        f"Jumlah kecamatan = {jumlah_kecamatan}, bukan 26. "
        "Periksa dataset luas sawah."
    )

if df_sawah["kecamatan"].duplicated().any():
    raise ValueError("Ada nama kecamatan yang duplikat.")

total_luas_sawah = df_sawah["luas_sawah_ha"].sum()

print(f"Jumlah kecamatan : {jumlah_kecamatan}")
print(f"Total luas sawah : {total_luas_sawah:,.2f} Ha")

if np.isclose(total_luas_sawah, 44066, atol=1):
    print("✓ Total luas sawah sesuai sekitar 44.066 Ha.")
else:
    print("⚠ Total luas sawah berbeda dari 44.066 Ha. Program tetap menggunakan total dari file.")


Jumlah kecamatan : 26
Total luas sawah : 44,066.00 Ha
✓ Total luas sawah sesuai sekitar 44.066 Ha.


In [ ]:
# ============================================================
# 8. HITUNG PROPORSI LUAS SAWAH
# ============================================================

df_sawah["proporsi_sawah"] = (
    df_sawah["luas_sawah_ha"] / total_luas_sawah
)

df_sawah["persentase_sawah"] = (
    df_sawah["proporsi_sawah"] * 100
)

total_proporsi = df_sawah["proporsi_sawah"].sum()

print(f"Total proporsi = {total_proporsi:.10f}")

if not np.isclose(total_proporsi, 1.0):
    raise ValueError("Total proporsi luas sawah tidak sama dengan 1.")

display(
    df_sawah[
        ["kecamatan", "luas_sawah_ha", "proporsi_sawah", "persentase_sawah"]
    ].sort_values("luas_sawah_ha", ascending=False)
)


Total proporsi = 1.0000000000


,kecamatan,luas_sawah_ha,proporsi_sawah,persentase_sawah
6,Wates,3370,0.0765,7.6476
16,Plemahan,3201,0.0726,7.2641
17,Purwoasri,2778,0.0630,6.3042
11,Kepung,2356,0.0535,5.3465
9,Gurah,2272,0.0516,5.1559
3,Kras,2191,0.0497,4.9721
14,Badas,2181,0.0495,4.9494
8,Plosoklaten,2153,0.0489,4.8859
5,Kandat,1984,0.0450,4.5023
20,Kayenkidul,1918,0.0435,4.3526


In [ ]:
# ============================================================
# 9. BENTUK 26 KECAMATAN × 5 KOMODITAS
# ============================================================

df_kecamatan = df_sawah[
    [
        "kecamatan",
        "luas_sawah_ha",
        "proporsi_sawah",
        "persentase_sawah",
    ]
].copy()

df_final = df_kecamatan.merge(df_tanaman, how="cross")

expected_rows = jumlah_kecamatan * len(df_tanaman)

if len(df_final) != expected_rows:
    raise ValueError(
        f"Jumlah baris {len(df_final)} tidak sesuai target {expected_rows}."
    )

print(
    f"✓ Dataset terbentuk: {jumlah_kecamatan} kecamatan × "
    f"{len(df_tanaman)} komoditas = {len(df_final)} baris"
)


✓ Dataset terbentuk: 26 kecamatan × 5 komoditas = 130 baris


In [ ]:
# ============================================================
# 10. LUAS PANEN DAN PRODUKSI
# ============================================================

df_final["luas_panen_ha"] = (
    df_final["luas_panen_kab_ha"] *
    df_final["proporsi_sawah"]
)

df_final["total_produksi_ton"] = (
    df_final["produksi_kab_ton"] *
    df_final["proporsi_sawah"]
)

# Ton/Ha -> Kuintal/Ha
df_final["produktivitas_kuintal_ha"] = np.where(
    df_final["luas_panen_ha"] > 0,
    (
        df_final["total_produksi_ton"] /
        df_final["luas_panen_ha"]
    ) * 10,
    np.nan,
)

display(df_final.head(10))


,kecamatan,luas_sawah_ha,proporsi_sawah,persentase_sawah,Tanaman,luas_panen_kab_ha,produksi_kab_ton,produktivitas_kab_kuintal_ha,luas_panen_ha,total_produksi_ton,produktivitas_kuintal_ha
0,Mojo,926,0.0210,2.1014,Padi,"34,198.2300","198,324.3200",57.9926,718.6393,"4,167.5741",57.9926
1,Mojo,926,0.0210,2.1014,Kedelai,1.9000,2.5030,13.1739,0.0399,0.0526,13.1739
2,Mojo,926,0.0210,2.1014,Kacang Tanah,476.5000,843.6965,17.7061,10.0131,17.7294,17.7061
3,Mojo,926,0.0210,2.1014,Ubi Jalar,538.5000,"15,825.8317",293.8873,11.3160,332.5630,293.8873
4,Mojo,926,0.0210,2.1014,Ketela Pohon,652.8000,"18,597.0245",284.8809,13.7179,390.7966,284.8809
5,Semen,1391,0.0316,3.1566,Padi,"34,198.2300","198,324.3200",57.9926,"1,079.5111","6,260.3624",57.9926
6,Semen,1391,0.0316,3.1566,Kedelai,1.9000,2.5030,13.1739,0.0600,0.0790,13.1739
7,Semen,1391,0.0316,3.1566,Kacang Tanah,476.5000,843.6965,17.7061,15.0413,26.6324,17.7061
8,Semen,1391,0.0316,3.1566,Ubi Jalar,538.5000,"15,825.8317",293.8873,16.9984,499.5627,293.8873
9,Semen,1391,0.0316,3.1566,Ketela Pohon,652.8000,"18,597.0245",284.8809,20.6065,587.0390,284.8809


In [ ]:
# ============================================================
# 11. METADATA DATA ESTIMASI
# ============================================================
kolom_final = [
    "kecamatan",
    "Tanaman",
    "luas_panen_ha",
    "total_produksi_ton",
    "produktivitas_kuintal_ha"
]

df_final = df_final[kolom_final]

display(df_final.head(15))


,kecamatan,Tanaman,luas_panen_ha,total_produksi_ton,produktivitas_kuintal_ha
0,Mojo,Padi,718.6393,"4,167.5741",57.9926
1,Mojo,Kedelai,0.0399,0.0526,13.1739
2,Mojo,Kacang Tanah,10.0131,17.7294,17.7061
3,Mojo,Ubi Jalar,11.3160,332.5630,293.8873
4,Mojo,Ketela Pohon,13.7179,390.7966,284.8809
5,Semen,Padi,"1,079.5111","6,260.3624",57.9926
6,Semen,Kedelai,0.0600,0.0790,13.1739
7,Semen,Kacang Tanah,15.0413,26.6324,17.7061
8,Semen,Ubi Jalar,16.9984,499.5627,293.8873
9,Semen,Ketela Pohon,20.6065,587.0390,284.8809


In [37]:
# ============================================================
# 12. CEK PRODUKTIVITAS
# ============================================================

cek_produktivitas = (
    df_final.groupby("Tanaman")["produktivitas_kuintal_ha"]
    .agg(
        minimum="min",
        maksimum="max",
        rata_rata="mean",
        standar_deviasi="std",
    )
    .reset_index()
)

print(
    "Catatan: dengan metode ini, produktivitas komoditas yang sama "
    "akan sama antar kecamatan."
)

display(cek_produktivitas)


Catatan: dengan metode ini, produktivitas komoditas yang sama akan sama antar kecamatan.


,Tanaman,minimum,maksimum,rata_rata,standar_deviasi
0,Kacang Tanah,17.7061,17.7061,17.7061,0.0000
1,Kedelai,13.1739,13.1739,13.1739,0.0000
2,Ketela Pohon,284.8809,284.8809,284.8809,0.0000
3,Padi,57.9926,57.9926,57.9926,0.0000
4,Ubi Jalar,293.8873,293.8873,293.8873,0.0000


In [ ]:
# ============================================================
# 13. EXPORT KE CSV
# ============================================================

OUTPUT_CSV = "Tanaman_Pangan_Kecamatan_Kediri.csv"

df_final.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)

print(f"✓ CSV berhasil dibuat: {OUTPUT_CSV}")


✓ CSV berhasil dibuat: Tanaman_Pangan_Kecamatan_Kediri.csv


In [ ]:
# ============================================================
# 14. TAMPILKAN HASIL AKHIR
# ============================================================

display(
    df_final[
        [
            "kecamatan",
            "Tanaman",
            "luas_panen_ha",
            "total_produksi_ton",
            "produktivitas_kuintal_ha",
        ]
    ].head(20)
)

print("\nFile output:")
print("-", OUTPUT_CSV)


,kecamatan,Tanaman,luas_panen_ha,total_produksi_ton,produktivitas_kuintal_ha
0,Mojo,Padi,718.6393,"4,167.5741",57.9926
1,Mojo,Kedelai,0.0399,0.0526,13.1739
2,Mojo,Kacang Tanah,10.0131,17.7294,17.7061
3,Mojo,Ubi Jalar,11.3160,332.5630,293.8873
4,Mojo,Ketela Pohon,13.7179,390.7966,284.8809
5,Semen,Padi,"1,079.5111","6,260.3624",57.9926
6,Semen,Kedelai,0.0600,0.0790,13.1739
7,Semen,Kacang Tanah,15.0413,26.6324,17.7061
8,Semen,Ubi Jalar,16.9984,499.5627,293.8873
9,Semen,Ketela Pohon,20.6065,587.0390,284.8809



File output:
- Tanaman_Pangan_Kecamatan_Kediri.csv
